In [ ]:
import pandas as pd
import ast
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
def preprocess_borg_data(file_path):
    # Load dataset
    df = pd.read_csv(file_path)
    
    # Function to safe-parse dictionary strings (e.g., "{'cpus': 0.02, ...}")
    def parse_dict_column(series, prefix):
        # Convert string representation of dict to actual dict
        dicts = series.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        # Flatten into new columns with a prefix
        return pd.json_normalize(dicts).add_prefix(f'{prefix}_')

    # Parse resource requests and actual usage
    req_df = parse_dict_column(df['resource_request'], 'req')
    usage_df = parse_dict_column(df['average_usage'], 'usage')

    # Join back to main dataframe and drop original complex columns
    df = pd.concat([df.drop(['resource_request', 'average_usage'], axis=1), req_df, usage_df], axis=1)

    # Feature Engineering: Resource Efficiency
    # Tasks nearing 1.0 (100% usage) are often high-risk for failure
    df['cpu_efficiency'] = df['usage_cpus'] / (df['req_cpus'] + 1e-9)
    df['mem_efficiency'] = df['usage_memory'] / (df['req_memory'] + 1e-9)
    
    # Drop high-cardinality/anonymized strings for simple modeling
    cols_to_drop = ['user', 'collection_name', 'collection_logical_name', 'machine_id']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # Handle missing values
    df = df.fillna(0)
    
    return df

In [ ]:
processed_df = preprocess_borg_data('borg_traces_data.csv')

In [ ]:
class FailurePredictorLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(FailurePredictorLSTM, self).__init__()
        self.hidden_size = hidden_size
        # The GRU keeps the 'internal state' of the task
        self.gru = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden=None):
        # x: [batch, seq_len, features]
        # out: [batch, seq_len, hidden_size]
        out, hidden = self.gru(x, hidden)
        
        # Predict failure probability for the latest time step
        prediction = self.sigmoid(self.classifier(out[:, -1, :]))
        return prediction, hidden

# Initialize model
# Features: priority, scheduling_class, req_cpus, req_mem, usage_cpus, usage_mem (6 total)
model = FailurePredictorLSTM(input_size=6)


In [ ]:
# Hyperparameters
input_size = 6   # Our 6 numeric features
hidden_size = 128
num_epochs = 300

# Initialize model, loss, and optimizer
model = FailurePredictorLSTM(input_size, hidden_size)
criterion = nn.BCELoss() # Binary Cross Entropy for failure (0 or 1)
optimizer = optim.Adam(model.parameters(), lr=0.001)


def train_with_detailed_metrics(model, loader, epochs=num_epochs, threshold=0.5):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss() # Binary Cross Entropy

    print(f"{'Epoch':<8} | {'Loss':<7} | {'Acc':<7} | {'Prec':<7} | {'FPR':<7} | {'FNR':<7}")
    print("-" * 65)

    for epoch in range(epochs):
        model.train()
        all_preds = []
        all_targets = []
        epoch_loss = 0

        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            
            # Forward pass
            probs, _ = model(batch_X)
            probs = probs.squeeze()
            
            loss = criterion(probs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()

            # Store predictions and targets for metrics
            # We use a threshold (default 0.5) to convert probabilities to 0 or 1
            preds = (probs > threshold).float()
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(batch_y.detach().cpu().numpy())

        # Calculate Final Metrics for the Epoch
        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)

        tp = np.sum((all_preds == 1) & (all_targets == 1))
        tn = np.sum((all_preds == 0) & (all_targets == 0))
        fp = np.sum((all_preds == 1) & (all_targets == 0))
        fn = np.sum((all_preds == 0) & (all_targets == 1))

        # Metrics Formulas
        accuracy = (tp + tn) / len(all_targets)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Error Rates
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0 # False Positive Rate (Type I)
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0 # False Negative Rate (Type II)

        # Print formatted results
        print(f"{epoch+1:<8} | {epoch_loss/len(loader):<7.4f} | {accuracy:<7.2%} | {precision:<7.2f} | {fpr:<7.2f} | {fnr:<7.2f}")

In [ ]:
processed_df

In [ ]:
def create_sliding_windows(df, window_size=5):
    X, y = [], []
    # Ensure tasks don't bleed into each other
    for cid, group in df.groupby('collection_id'):
        # Sort by time if your 'time' column isn't already sorted
        data = group[['priority', 'scheduling_class', 'req_cpus', 'req_memory', 'usage_cpus', 'usage_memory']].values
        targets = group['failed'].values
        
        # Create windows: use rows [i : i+5] to predict row [i+5]
        for i in range(len(data) - window_size):
            X.append(data[i : i + window_size])
            y.append(targets[i + window_size])
            
    return torch.tensor(np.array(X)).float(), torch.tensor(np.array(y)).float()

# Example usage with your preprocessed dataframe
X_train, y_train = create_sliding_windows(processed_df)
dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
train_with_detailed_metrics(model, train_loader)

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}
torch.save(checkpoint, 'lstm.pth')

In [ ]:
checkpoint = torch.load('lstm.pth')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [ ]:
train_with_detailed_metrics(model, train_loader, epochs=1)